# LeetCode 2506 — Count Pairs Of Similar Strings bằng Python

Link đề: https://leetcode.com/problems/count-pairs-of-similar-strings/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, String, Bit Manipulation, Counting

Skeleton LeetCode:

```python
class Solution:
    def similarPairs(self, words: List[str]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai chuỗi là **giống nhau (similar)** khi chúng được tạo bởi **cùng một tập ký tự** (không quan tâm số lần xuất hiện hay thứ tự); đếm số cặp chỉ số `(i, j)`, `i < j`, mà `words[i]` và `words[j]` giống nhau.
2. Dùng `set` để biểu diễn "tập ký tự" của một chuỗi, và **bitmask** 26 bit như một cách mã hoá tập ký tự bằng một số nguyên.
3. Nhận ra bài toán "đếm cặp có cùng đặc trưng" → **gom nhóm theo khoá** (`Counter`) rồi cộng `C(c, 2)`; hoặc đếm "online" `ans += seen[key]`.
4. Viết 5 cách: so sánh từng cặp bằng `set`, so sánh từng cặp bằng mask đã tính trước, `Counter` các mask + `comb`, đếm online một lượt, và khoá chuỗi `sorted(set(w))`; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: nhầm với anagram, tạo mask bằng `+=` thay vì `|=`, đếm số nhóm thay vì số cặp, chỉ so sánh số ký tự khác nhau.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `words` (chỉ chữ cái thường). Hai chuỗi **giống nhau** nếu tập các ký tự xuất hiện trong chúng **bằng nhau**. Ví dụ `"abca"` và `"cba"` giống nhau vì cùng là tập `{a, b, c}`; còn `"abacaba"` và `"bcfd"` thì không.

Trả về số cặp `(i, j)` thoả `0 <= i < j <= words.length - 1` và `words[i]`, `words[j]` giống nhau.

| Ví dụ | words | Output | Giải thích |
|---|---|---|---|
| 1 | `["aba","aabb","abcd","bac","aabc"]` | 2 | cặp `(0,1)` (cùng `{a,b}`) và cặp `(3,4)` (cùng `{a,b,c}`) |
| 2 | `["aabb","ab","ba"]` | 3 | cả ba chuỗi cùng `{a,b}`: các cặp `(0,1)`, `(0,2)`, `(1,2)` |
| 3 | `["nba","cba","dba"]` | 0 | ba tập khác nhau |

Constraints: `1 <= words.length <= 100`; `1 <= words[i].length <= 100`; `words[i]` chỉ gồm chữ cái thường tiếng Anh.

In [1]:
words = ["aba", "aabb", "abcd", "bac", "aabc"]
for w in words:
    print(f"{w!r:7} -> tập ký tự {sorted(set(w))}")
pairs = [(i, j) for i in range(len(words)) for j in range(i + 1, len(words)) if set(words[i]) == set(words[j])]
print("các cặp giống nhau:", pairs, "->", len(pairs))

'aba'   -> tập ký tự ['a', 'b']
'aabb'  -> tập ký tự ['a', 'b']
'abcd'  -> tập ký tự ['a', 'b', 'c', 'd']
'bac'   -> tập ký tự ['a', 'b', 'c']
'aabc'  -> tập ký tự ['a', 'b', 'c']
các cặp giống nhau: [(0, 1), (3, 4)] -> 2


## 3. Kiến thức nền

### 3.1. `set` — tập hợp các ký tự

`set("aabb")` cho `{'a', 'b'}`: chỉ giữ mỗi ký tự **một lần**, không có thứ tự. Hai set so sánh bằng `==` khi chứa **cùng các phần tử**. Đây chính là khái niệm "similar" của đề.

In [2]:
print(set("aabb"), set("ba"))
print(set("aabb") == set("ba"), set("aba") == set("abcd"))
print(len(set("abacaba")))

{'a', 'b'} {'a', 'b'}
True False
3


### 3.2. Bitmask 26 bit cho tập ký tự

Có 26 chữ cái thường, nên một tập ký tự là một số 26 bit: bit thứ `k` bật nếu chữ cái thứ `k` có mặt. `1 << k` là số có đúng bit `k`; `mask |= 1 << k` **bật** bit đó (dùng `|=`, **không phải** `+=`). `ord(c) - ord('a')` đổi chữ cái thành `0..25`. Hai tập bằng nhau ⇔ hai mask bằng nhau.

In [3]:
def to_mask(w):
    mask = 0
    for ch in w:
        mask |= 1 << (ord(ch) - ord("a"))
    return mask

for w in ["aba", "aabb", "abcd", "bac", "aabc"]:
    print(f"{w!r:7} -> mask {to_mask(w):>2} = {to_mask(w):04b}")

'aba'   -> mask  3 = 0011
'aabb'  -> mask  3 = 0011
'abcd'  -> mask 15 = 1111
'bac'   -> mask  7 = 0111
'aabc'  -> mask  7 = 0111


### 3.3. `collections.Counter` và `math.comb`

`Counter(keys)` đếm số lần mỗi khoá xuất hiện. Nếu có `c` chuỗi cùng khoá thì số cặp giữa chúng là `C(c, 2) = c * (c - 1) / 2` (`math.comb(c, 2)`): 3 chuỗi → 3 cặp, 4 chuỗi → 6 cặp.

In [4]:
from collections import Counter
import math
cnt = Counter([3, 3, 15, 7, 7])
print(cnt)
print({m: math.comb(c, 2) for m, c in cnt.items()})
print([math.comb(c, 2) for c in range(1, 6)])

Counter({3: 2, 7: 2, 15: 1})
{3: 1, 15: 0, 7: 1}
[0, 1, 3, 6, 10]


### 3.4. Đếm "online" với dict: `ans += seen[key]`

Duyệt từng phần tử theo thứ tự; trước khi thêm phần tử hiện tại vào `seen`, số phần tử **đã gặp** cùng khoá chính là số cặp mới nó tạo với các phần tử trước. Cộng dồn số này cho ra tổng số cặp mà không cần `comb`. `dict.get(key, 0)` hoặc `defaultdict(int)` tránh lỗi khoá chưa có.

In [5]:
seen = {}
ans = 0
for key in [3, 3, 15, 7, 7]:
    ans += seen.get(key, 0)
    seen[key] = seen.get(key, 0) + 1
    print(f"key={key:>2}: seen={seen}, ans={ans}")

key= 3: seen={3: 1}, ans=0
key= 3: seen={3: 2}, ans=1
key=15: seen={3: 2, 15: 1}, ans=1
key= 7: seen={3: 2, 15: 1, 7: 1}, ans=1
key= 7: seen={3: 2, 15: 1, 7: 2}, ans=2


### 3.5. `itertools.combinations` — mọi cặp `i < j`

`combinations(range(n), 2)` sinh mọi cặp `(i, j)` với `i < j`, đúng thứ tự không lặp. Gọn hơn hai vòng `for`.

In [6]:
from itertools import combinations
print(list(combinations(range(4), 2)))
print(len(list(combinations(range(100), 2))))

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
4950


### 3.6. Khoá hashable: `frozenset` và `"".join(sorted(set(w)))`

`set` có thể thay đổi nên **không** dùng được làm khoá của dict. Có hai cách chuẩn hoá thành khoá hashable: `frozenset(w)` (set bất biến) hoặc chuỗi các chữ cái **không trùng, sắp xếp** `"".join(sorted(set(w)))`. Cả hai giống nhau với hai chuỗi giống nhau (chuỗi cùng tập ký tự cho cùng chuỗi chuẩn).

In [7]:
print(frozenset("aabb") == frozenset("ba"))
print("".join(sorted(set("aabb"))), "".join(sorted(set("ba"))), "".join(sorted(set("bac"))))
d = {frozenset("ab"): 1}
print(d[frozenset("ba")])

True
ab ab abc
1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Cách tự nhiên nhất: thử mọi cặp `(i, j)` và so sánh hai tập ký tự.

Ví dụ 1 với `["aba","aabb","abcd","bac","aabc"]`:

```
tập ký tự: 0:{a,b}  1:{a,b}  2:{a,b,c,d}  3:{a,b,c}  4:{a,b,c}
cặp giống nhau: (0,1) vì cùng {a,b};  (3,4) vì cùng {a,b,c}
mọi cặp khác đều khác tập -> 2
```

Nhận xét quan trọng: "giống nhau" là một quan hệ **cùng đặc trưng** (đặc trưng = tập ký tự). Với những bài như vậy, ta không cần so sánh từng cặp: chỉ cần tính "đặc trưng" của từng chuỗi **một lần**, rồi đếm xem mỗi đặc trưng xuất hiện bao nhiêu lần.

## 5. Cách 1 — Brute Force / baseline

Hai vòng `for` chọn `i < j`; với mỗi cặp tạo hai `set` rồi so sánh. Đúng định nghĩa của đề nên cũng là **oracle** của các test sau.

Nút thắt: có `n(n-1)/2` cặp, mỗi cặp tạo hai set tốn `O(L)` → `O(n^2 * L)`. Ta đang tính lại `set(words[i])` nhiều lần cho cùng một `i`.

In [8]:
class SolutionBrute:
    def similarPairs(self, words):
        n = len(words)
        total = 0
        for i in range(n):
            for j in range(i + 1, n):
                if set(words[i]) == set(words[j]):
                    total += 1
        return total

print(SolutionBrute().similarPairs(["aba", "aabb", "abcd", "bac", "aabc"]))
print(SolutionBrute().similarPairs(["aabb", "ab", "ba"]))
print(SolutionBrute().similarPairs(["nba", "cba", "dba"]))

2
3
0


**Độ phức tạp:** Time `O(n^2 * L)` (`L` là độ dài chuỗi), Space `O(L)`. Với `n <= 100`, `L <= 100` còn chạy được nhưng phí.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** mỗi chuỗi → **một khoá** (mask 26 bit). Hai chuỗi giống nhau ⇔ cùng khoá. Vậy chỉ cần:

1. Tính khoá của mỗi chuỗi **một lần** (`O(L)`).
2. Gom theo khoá và đếm: nhóm có `c` chuỗi đóng góp `C(c, 2)` cặp.

Hoặc đếm online: khi duyệt chuỗi mới, `ans += seen[key]` rồi tăng `seen[key]`.

**Quy tắc "nếu… thì…":**

- Nếu hai chuỗi cùng mask → chúng là một cặp giống nhau.
- Nếu nhóm có `c` chuỗi → `c * (c - 1) / 2` cặp (mỗi cặp đếm **một** lần, `i < j`).
- Nếu nhóm chỉ có một chuỗi → `0` cặp.

Invariant (đếm online): trước khi xử lý chuỗi thứ `k`, `seen[key]` bằng số chuỗi trong `k` chuỗi đầu có khoá `key`, và `ans` bằng số cặp giống nhau trong `k` chuỗi đầu.

In [9]:
words = ["aba", "aabb", "abcd", "bac", "aabc"]
seen = {}
ans = 0
for w in words:
    key = to_mask(w)
    ans += seen.get(key, 0)
    seen[key] = seen.get(key, 0) + 1
print("seen:", seen, "-> ans =", ans)

seen: {3: 2, 15: 1, 7: 2} -> ans = 2


## 7. Các cách tiếp theo

Ta có 5 cách: (1) mọi cặp với `set`; (2) mọi cặp với mask tính sẵn; (3) `Counter` mask + `comb`; (4) đếm online một lượt; (5) khoá chuỗi `sorted(set(w))`.

### Cách 2 — Tính mask trước, so sánh từng cặp

Vẫn `O(n^2)` cặp nhưng mỗi so sánh chỉ là hai số nguyên `O(1)`; phần tạo mask tốn `O(n * L)` **một lần**. Rẻ hơn Cách 1 rất nhiều khi `L` lớn.

In [10]:
class SolutionMaskPairs:
    def similarPairs(self, words):
        masks = []
        for w in words:
            m = 0
            for ch in w:
                m |= 1 << (ord(ch) - ord("a"))
            masks.append(m)
        n = len(words)
        total = 0
        for i in range(n):
            for j in range(i + 1, n):
                if masks[i] == masks[j]:
                    total += 1
        return total

print(SolutionMaskPairs().similarPairs(["aba", "aabb", "abcd", "bac", "aabc"]))
print(SolutionMaskPairs().similarPairs(["aabb", "ab", "ba"]))
print(SolutionMaskPairs().similarPairs(["nba", "cba", "dba"]))

2
3
0


Độ phức tạp Cách 2: Time `O(n * L + n^2)`, Space `O(n)`.

### Cách 3 — `Counter` các mask + `comb`

Đếm số chuỗi của mỗi mask bằng `Counter`, rồi cộng `comb(c, 2)` cho từng nhóm. Không còn vòng lặp `O(n^2)`.

In [11]:
class SolutionCounter:
    def similarPairs(self, words):
        cnt = Counter()
        for w in words:
            m = 0
            for ch in w:
                m |= 1 << (ord(ch) - ord("a"))
            cnt[m] += 1
        return sum(math.comb(c, 2) for c in cnt.values())

print(SolutionCounter().similarPairs(["aba", "aabb", "abcd", "bac", "aabc"]))
print(SolutionCounter().similarPairs(["aabb", "ab", "ba"]))
print(SolutionCounter().similarPairs(["nba", "cba", "dba"]))

2
3
0


Độ phức tạp Cách 3: Time `O(n * L)`, Space `O(k)` (`k` = số nhóm mask phân biệt, `k <= n`).

### Cách 4 — Đếm online một lượt

Không cần `comb`: mỗi chuỗi mới cộng thêm số chuỗi trước đó có cùng mask. Tổng số cặp đúng là tổng các số đó. Cách này mở rộng được cho bài "đếm cặp thoả điều kiện" khác.

In [12]:
class SolutionOnline:
    def similarPairs(self, words):
        seen = {}
        total = 0
        for w in words:
            m = 0
            for ch in w:
                m |= 1 << (ord(ch) - ord("a"))
            total += seen.get(m, 0)
            seen[m] = seen.get(m, 0) + 1
        return total

print(SolutionOnline().similarPairs(["aba", "aabb", "abcd", "bac", "aabc"]))
print(SolutionOnline().similarPairs(["aabb", "ab", "ba"]))
print(SolutionOnline().similarPairs(["nba", "cba", "dba"]))

2
3
0


Độ phức tạp Cách 4: Time `O(n * L)`, Space `O(k)`.

### Cách 5 — Khoá chuỗi `"".join(sorted(set(w)))`

Không dùng bit: chuẩn hoá mỗi chuỗi thành chuỗi các chữ cái **phân biệt, sắp xếp**; hai chuỗi giống nhau ⇔ cùng chuỗi chuẩn. Dễ đọc, hơi chậm hơn mask nhưng không cần thao tác bit.

In [13]:
class SolutionSortedKey:
    def similarPairs(self, words):
        cnt = Counter("".join(sorted(set(w))) for w in words)
        return sum(c * (c - 1) // 2 for c in cnt.values())

print(SolutionSortedKey().similarPairs(["aba", "aabb", "abcd", "bac", "aabc"]))
print(SolutionSortedKey().similarPairs(["aabb", "ab", "ba"]))
print(SolutionSortedKey().similarPairs(["nba", "cba", "dba"]))

2
3
0


Độ phức tạp Cách 5: Time `O(n * (L + 26 log 26))` ≈ `O(n * L)` (sắp xếp tối đa 26 chữ cái), Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 4 (đếm online) với `["aba", "aabb", "abcd", "bac", "aabc"]`:

| bước | chuỗi | mask (nhị phân) | `seen` trước | cộng thêm | `ans` |
|---|---|---|---|---|---|
| 1 | `aba` | 3 (`0011`) | `{}` | 0 | 0 |
| 2 | `aabb` | 3 (`0011`) | `{3: 1}` | 1 | 1 |
| 3 | `abcd` | 15 (`1111`) | `{3: 2}` | 0 | 1 |
| 4 | `bac` | 7 (`0111`) | `{3: 2, 15: 1}` | 0 | 1 |
| 5 | `aabc` | 7 (`0111`) | `{3: 2, 15: 1, 7: 1}` | 1 | 2 |

Kết quả `2`.

In [14]:
def trace_online(words):
    seen = {}
    ans = 0
    for step, w in enumerate(words, start=1):
        m = to_mask(w)
        add = seen.get(m, 0)
        print(f"bước {step}: {w!r:7} mask={m:>2} ({m:04b}) seen trước={seen} cộng thêm {add}", end="")
        ans += add
        seen[m] = seen.get(m, 0) + 1
        print(f" -> ans={ans}")
    return ans

print("kết quả:", trace_online(["aba", "aabb", "abcd", "bac", "aabc"]))
print("kết quả:", trace_online(["aabb", "ab", "ba"]))

bước 1: 'aba'   mask= 3 (0011) seen trước={} cộng thêm 0 -> ans=0
bước 2: 'aabb'  mask= 3 (0011) seen trước={3: 1} cộng thêm 1 -> ans=1
bước 3: 'abcd'  mask=15 (1111) seen trước={3: 2} cộng thêm 0 -> ans=1
bước 4: 'bac'   mask= 7 (0111) seen trước={3: 2, 15: 1} cộng thêm 0 -> ans=1
bước 5: 'aabc'  mask= 7 (0111) seen trước={3: 2, 15: 1, 7: 1} cộng thêm 1 -> ans=2
kết quả: 2
bước 1: 'aabb'  mask= 3 (0011) seen trước={} cộng thêm 0 -> ans=0
bước 2: 'ab'    mask= 3 (0011) seen trước={3: 1} cộng thêm 1 -> ans=1
bước 3: 'ba'    mask= 3 (0011) seen trước={3: 2} cộng thêm 2 -> ans=3
kết quả: 3


## 9. Code nộp LeetCode

In [15]:
from typing import List
from collections import defaultdict

class Solution:
    def similarPairs(self, words: List[str]) -> int:
        seen = defaultdict(int)
        total = 0
        for w in words:
            mask = 0
            for ch in w:
                mask |= 1 << (ord(ch) - ord("a"))
            total += seen[mask]
            seen[mask] += 1
        return total

print(Solution().similarPairs(["aba", "aabb", "abcd", "bac", "aabc"]),
      Solution().similarPairs(["aabb", "ab", "ba"]),
      Solution().similarPairs(["nba", "cba", "dba"]))

2 3 0


## 10. Vì sao thuật toán đúng

- **Mask xác định tập ký tự:** bit `k` của mask bật ⇔ chữ cái thứ `k` có mặt trong chuỗi (phép `|=` chỉ bật bit, **không đếm số lần**). Vậy hai chuỗi có cùng mask ⇔ cùng tập ký tự ⇔ giống nhau theo đề. Đây là song ánh giữa tập con của 26 chữ cái và số 26 bit.
- **Đếm online:** khi xử lý chuỗi thứ `j` (đếm từ 0), `seen[mask]` bằng số chỉ số `i < j` có cùng mask. Cộng số đó chính là số cặp `(i, j)` với `i < j` kết thúc tại `j`. Mỗi cặp `(i, j)` có `i < j` được đếm đúng một lần (tại `j`).
- **`Counter` + `comb`:** trong nhóm `c` chuỗi cùng mask, mọi cặp hai chuỗi đều giống nhau, và có `C(c, 2)` cặp không thứ tự (tương ứng một-một với cặp `i < j`). Các nhóm khác mask không có cặp chéo.
- **Cách 5:** `"".join(sorted(set(w)))` là biểu diễn chuẩn tắc duy nhất của tập ký tự (sắp xếp loại bỏ thứ tự, `set` loại bỏ số lần), nên hai chuỗi giống nhau ⇔ cùng khoá chuỗi.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(words)`, `L` = độ dài chuỗi lớn nhất, `k` = số nhóm khoá phân biệt (`k <= n`).

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp, tạo `set` mỗi lần | `O(n^2 * L)` | `O(L)` |
| 2. Mask tính sẵn, mọi cặp | `O(n * L + n^2)` | `O(n)` |
| 3. `Counter` mask + `comb` | `O(n * L)` | `O(k)` |
| 4. Đếm online một lượt | `O(n * L)` | `O(k)` |
| 5. Khoá `sorted(set(w))` | `O(n * L)` | `O(n)` |

Với `n, L <= 100` mọi cách đều nhanh. Cách 3, 4 là chuẩn tuyến tính theo tổng độ dài các chuỗi (phải đọc hết đầu vào ít nhất một lần).

## 12. Bộ test

Test 1: ví dụ đề và các edge case (`n = 1`, tất cả giống nhau, không có cặp nào, chuỗi một chữ cái, hai chuỗi khác nhau nhưng cùng độ dài tập).

In [16]:
impls = [SolutionBrute(), SolutionMaskPairs(), SolutionCounter(), SolutionOnline(), SolutionSortedKey(), Solution()]
tests = [
    (["aba", "aabb", "abcd", "bac", "aabc"], 2),
    (["aabb", "ab", "ba"], 3),
    (["nba", "cba", "dba"], 0),
    (["a"], 0),                              # n = 1: không có cặp
    (["a", "a"], 1),
    (["a", "aa", "aaa", "aaaa"], 6),         # cả bốn cùng {a}: C(4,2)
    (["ab", "cd"], 0),                       # cùng số ký tự khác nhau nhưng tập khác (bắt lỗi len(set))
    (["aa", "b"], 0),                        # bắt lỗi dùng += thay |= (mask 2 = mask 2)
    (["ab", "ba", "abab", "baba"], 6),       # đều là {a,b}
    (["abc", "cab", "bca", "abcd"], 3),      # ba chuỗi cùng {a,b,c}, một khác
    (["aabb", "ab", "ba"], 3),               # anagram chỉ cho 1 cặp (bắt lỗi nhầm anagram)
    (["z", "y", "x", "z"], 1),               # chỉ hai chuỗi "z" giống nhau
    (["abc", "abc", "abd", "abd", "abd"], 1 + 3),   # nhóm 2 -> 1 cặp, nhóm 3 -> 3 cặp
    (["ab", "ab", "ab", "cd", "cd"], 3 + 1),        # nhóm 3 -> 3 cặp, nhóm 2 -> 1 cặp (bắt lỗi đếm số nhóm)
]
for words, exp in tests:
    assert 1 <= len(words) <= 100 and all(1 <= len(w) <= 100 and w.islower() and w.isalpha() for w in words)
    for im in impls:
        assert im.similarPairs(list(words)) == exp, (type(im).__name__, words, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 14


Test 2: ngẫu nhiên có seed, đúng constraints (`1 <= n <= 100`, `1 <= len(w) <= 100`, chữ cái thường). Dùng bảng chữ cái nhỏ để có nhiều cặp trùng. Oracle độc lập dựa trên `itertools.combinations` + `frozenset`.

In [17]:
import random, string
random.seed(2506)

def oracle(words):
    return sum(1 for a, b in combinations(words, 2) if frozenset(a) == frozenset(b))

for t in range(1200):
    n = random.randint(1, 100) if t % 20 == 0 else random.randint(1, 12)
    alpha = string.ascii_lowercase[: random.choice([1, 2, 3, 4, 26])]
    max_len = 100 if t % 20 == 0 else 8
    words = ["".join(random.choice(alpha) for _ in range(random.randint(1, max_len))) for _ in range(n)]
    assert 1 <= len(words) <= 100 and all(1 <= len(w) <= 100 for w in words)
    exp = oracle(words)
    for im in impls:
        assert im.similarPairs(list(words)) == exp, (type(im).__name__, words)
print("1200 test ngẫu nhiên OK")

1200 test ngẫu nhiên OK


Test 3: tính chất — đáp án nằm trong `0..C(n, 2)`; hoán vị mảng và hoán vị các ký tự trong từng chuỗi không đổi đáp án; không sửa input.

In [18]:
random.seed(11)
for _ in range(300):
    n = random.randint(1, 20)
    words = ["".join(random.choice("abc") for _ in range(random.randint(1, 6))) for _ in range(n)]
    base = Solution().similarPairs(words)
    assert 0 <= base <= math.comb(n, 2)
    shuffled = list(words)
    random.shuffle(shuffled)
    assert Solution().similarPairs(shuffled) == base
    scrambled = ["".join(random.sample(w, len(w))) for w in words]
    assert Solution().similarPairs(scrambled) == base
    snapshot = list(words)
    Solution().similarPairs(words)
    assert words == snapshot
print("tính chất (khoảng giá trị, bất biến theo hoán vị mảng và ký tự, không sửa input): OK")

tính chất (khoảng giá trị, bất biến theo hoán vị mảng và ký tự, không sửa input): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: nhầm "giống nhau" với "anagram" (so sánh chuỗi đã sắp xếp)

In [19]:
def wrong_anagram(words):
    cnt = Counter("".join(sorted(w)) for w in words)       # SAI: giữ nguyên số lần xuất hiện
    return sum(c * (c - 1) // 2 for c in cnt.values())

words = ["aabb", "ab", "ba"]
print("sai :", wrong_anagram(words))
print("đúng:", Solution().similarPairs(words))
assert wrong_anagram(words) != Solution().similarPairs(words)

sai : 1
đúng: 3


Vì sao sai: anagram yêu cầu **cùng số lần** từng ký tự; đề chỉ cần **cùng tập ký tự**. `"aabb"` và `"ab"` giống nhau (cùng `{a,b}`) nhưng không là anagram. **Sửa:** dùng `set` (loại số lần) trước khi chuẩn hoá.

### Lỗi 2: tạo mask bằng `+=` thay vì `|=`

In [20]:
def wrong_plus_mask(words):
    seen = {}
    total = 0
    for w in words:
        m = 0
        for ch in w:
            m += 1 << (ord(ch) - ord("a"))      # SAI: phải là |= (bật bit), += cộng dồn gây tràn bit
        total += seen.get(m, 0)
        seen[m] = seen.get(m, 0) + 1
    return total

words = ["aa", "b"]
print("sai :", wrong_plus_mask(words))
print("đúng:", Solution().similarPairs(words))
assert wrong_plus_mask(words) != Solution().similarPairs(words)

sai : 1
đúng: 0


Vì sao sai: `"aa"` → `1 + 1 = 2`, bằng đúng mask của `"b"` (`1 << 1 = 2`), nên hai chuỗi khác tập bị coi là giống nhau. Phép `|=` **bật** bit và giữ nguyên nếu đã bật (idempotent); `+=` cộng dồn nên mất tính chất "tập hợp". **Sửa:** `m |= 1 << k`.

### Lỗi 3: đếm số nhóm giống nhau thay vì số cặp

In [21]:
def wrong_count_groups(words):
    cnt = Counter(frozenset(w) for w in words)
    return sum(1 for c in cnt.values() if c >= 2)      # SAI: mỗi nhóm đóng góp C(c, 2), không phải 1

words = ["aabb", "ab", "ba"]
print("sai :", wrong_count_groups(words))
print("đúng:", Solution().similarPairs(words))
assert wrong_count_groups(words) != Solution().similarPairs(words)

sai : 1
đúng: 3


Vì sao sai: nhóm 3 chuỗi giống nhau tạo ra `C(3, 2) = 3` cặp, không phải 1. Đề đếm **cặp chỉ số**. **Sửa:** cộng `c * (c - 1) // 2` hoặc đếm online `ans += seen[key]`.

### Lỗi 4: chỉ so sánh số ký tự khác nhau (`len(set(...))`)

In [22]:
def wrong_len_set(words):
    n = len(words)
    return sum(1 for i in range(n) for j in range(i + 1, n) if len(set(words[i])) == len(set(words[j])))   # SAI

words = ["ab", "cd"]
print("sai :", wrong_len_set(words))
print("đúng:", Solution().similarPairs(words))
assert wrong_len_set(words) != Solution().similarPairs(words)

sai : 1
đúng: 0


Vì sao sai: hai tập có **cùng kích thước** chưa chắc **bằng nhau** (`{a,b}` và `{c,d}` đều 2 phần tử). Phải so sánh nội dung tập (hoặc mask), không chỉ kích thước.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm cặp `(i < j)` có cùng một đặc trưng" → tính **khoá** cho từng phần tử rồi gom nhóm (`Counter`) hoặc đếm online `ans += seen[key]`.
- Đặc trưng là "tập ký tự" → bitmask 26 bit (`mask |= 1 << (ord(c) - 97)`) hoặc `frozenset`/chuỗi chuẩn tắc.
- Anagram → khoá là chuỗi sắp xếp hoặc mảng đếm 26; **tập** ký tự → khoá bỏ qua số lần xuất hiện.
- Số cặp trong nhóm `c`: `c * (c - 1) // 2`; online cộng `seen[key]` trước rồi mới tăng.
- Dùng `defaultdict(int)` hoặc `dict.get(key, 0)` để tránh `KeyError`.
- `n <= 100` thì brute force `O(n^2 * L)` vẫn qua; nhưng hãy rèn thói quen "tính khoá một lần".

## 15. Biến thể của bài

1. Hai chuỗi giống nhau nếu **cùng số lần xuất hiện từng ký tự** (anagram) — Group Anagrams (LeetCode 49) và Valid Anagram.
2. Đếm cặp mà tập ký tự của chuỗi này **là tập con** của chuỗi kia (so sánh mask bằng `a & b == a`).
3. Trả về các **nhóm** chuỗi giống nhau thay vì số cặp (Group Anagrams dạng tập).
4. Đếm số cặp `(i, j)` mà hai chuỗi có **ít nhất một ký tự chung** (`mask_i & mask_j != 0`).
5. Cho mảng rất lớn (`n = 10^5`): bắt buộc dùng cách 3/4 (tuyến tính), không dùng mọi cặp.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp + `set` | so sánh hai set mỗi cặp | `O(n^2 * L)` | `O(L)` | hiểu đề, oracle test |
| 2. Mọi cặp + mask sẵn | mask một lần, so sánh `O(1)` | `O(nL + n^2)` | `O(n)` | `n` nhỏ, muốn code ngắn |
| 3. `Counter` mask + `comb` | gom nhóm theo mask | `O(nL)` | `O(k)` | **chuẩn**, dễ giải thích |
| 4. Đếm online | `ans += seen[mask]` một lượt | `O(nL)` | `O(k)` | pattern đếm cặp tổng quát |
| 5. Khoá `sorted(set(w))` | chuỗi chuẩn tắc thay mask | `O(nL)` | `O(n)` | không muốn dùng bit |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `similarPairs(["xy", "yx", "xxyy", "z"])` bằng bao nhiêu?

**Bài 2 (tính tay):** với `["ab", "ba", "abc", "cab", "bca", "d"]`, tính mask của từng chuỗi rồi đếm các cặp.

**Bài 3 (điền TODO):** hoàn thành `char_mask(w)` trả về mask 26 bit của chuỗi.

**Bài 4 (tự viết lại):** viết `group_similar(words)` trả về dict `{mask: [các chỉ số]}` các chuỗi cùng tập ký tự.

**Bài 5 (mini challenge):** viết `count_pairs_common_char(words)` đếm cặp `(i < j)` có **ít nhất một ký tự chung** (gợi ý: `mask_i & mask_j != 0`).

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().similarPairs(["xy", "yx", "xxyy", "z"]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def char_mask(w):
    mask = 0
    # TODO: với mỗi ký tự ch trong w, bật bit (ord(ch) - ord('a')) bằng |=
    return mask

3


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** hai chuỗi giống nhau ⇔ cùng **tập** ký tự; số lần xuất hiện không quan trọng.

**Mức 2 (vừa):** Bài 3: `mask |= 1 << (ord(ch) - ord("a"))`. Bài 4: duyệt `enumerate(words)`, `groups.setdefault(char_mask(w), []).append(i)`. Bài 5: hai vòng `for i < j` và kiểm tra `masks[i] & masks[j]`.

**Mức 3 (gần đáp án):** Bài 1: `"xy"`, `"yx"`, `"xxyy"` cùng `{x,y}` (3 chuỗi → 3 cặp), `"z"` riêng → `3`. Bài 2: `ab`,`ba` cùng mask 3 → 1 cặp; `abc`,`cab`,`bca` cùng mask 7 → 3 cặp; `d` riêng → tổng `4`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1: {x,y} x3 -> 3 cặp ; {z} -> 0
assert Solution().similarPairs(["xy", "yx", "xxyy", "z"]) == 3

# Bài 2: ab, ba -> 1 cặp ; abc, cab, bca -> 3 cặp ; d riêng
words2 = ["ab", "ba", "abc", "cab", "bca", "d"]
assert [to_mask(w) for w in words2] == [3, 3, 7, 7, 7, 8]
assert Solution().similarPairs(words2) == 1 + 3 == 4

# Bài 3
def char_mask(w):
    mask = 0
    for ch in w:
        mask |= 1 << (ord(ch) - ord("a"))
    return mask
assert char_mask("aba") == 3
assert char_mask("abcd") == 15
assert char_mask("z") == 1 << 25

# Bài 4
def group_similar(words):
    groups = {}
    for i, w in enumerate(words):
        groups.setdefault(char_mask(w), []).append(i)
    return groups
assert group_similar(["aba", "aabb", "abcd", "bac", "aabc"]) == {3: [0, 1], 15: [2], 7: [3, 4]}
assert group_similar(["a"]) == {1: [0]}

# Bài 5
def count_pairs_common_char(words):
    masks = [char_mask(w) for w in words]
    n = len(words)
    return sum(1 for i in range(n) for j in range(i + 1, n) if masks[i] & masks[j])
assert count_pairs_common_char(["nba", "cba", "dba"]) == 3      # cả ba cùng chứa 'a' và 'b'
assert count_pairs_common_char(["ab", "cd", "ae"]) == 1         # chỉ (ab, ae) chung 'a'
assert count_pairs_common_char(["a", "b", "c"]) == 0
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm cặp cùng khoá" (online):**

```python
from collections import defaultdict

seen = defaultdict(int)
total = 0
for w in words:
    key = 0
    for ch in w:
        key |= 1 << (ord(ch) - ord("a"))   # tập ký tự -> mask
    total += seen[key]                      # cộng số phần tử đã gặp cùng khoá
    seen[key] += 1
return total
```

**Checklist trước khi submit:**

- [ ] Hiểu "giống nhau" = cùng **tập** ký tự (không phải anagram)?
- [ ] Tạo mask bằng `|=` (không phải `+=`)?
- [ ] Cộng `seen[key]` **trước** khi tăng `seen[key]` (đếm cặp `i < j`, không đếm cả bản thân)?
- [ ] Nếu gom nhóm: dùng `c * (c - 1) // 2` chứ không phải số nhóm?
- [ ] Đã thử `n = 1`, tất cả giống nhau, `["aa", "b"]`, `["ab", "cd"]`?
- [ ] Đã đối chiếu các cách bằng test ngẫu nhiên có seed?
- [ ] Chữ ký đúng `similarPairs(self, words: List[str]) -> int`?